# PHASE 5A: Traditional Modeling and Validation (KNN, SVM, Random Forest, LightGBM)

## Objective
Train and compare classical ML models for RUL regression, aligned with:
- **Source A**: KNN / SVM / RandomForest training patterns and visual diagnostics.
- **Source B**: LightGBM model training and search-based tuning.
- **Comprehensive Plan**: RMSE + NASA score prioritization with reproducible validation.

## Inputs
- `../data/experiments/DS-B/train.csv`
- `../data/experiments/DS-B/valid.csv`
- `../data/experiments/DS-B/test.csv`


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, GroupKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

import lightgbm as lgb

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120})

DATA_DIR = Path('../data/experiments/DS-B')
TRAIN_PATH = DATA_DIR / 'train.csv'
VALID_PATH = DATA_DIR / 'valid.csv'
TEST_PATH = DATA_DIR / 'test.csv'
TARGET = 'RUL_clipped'
ID_COL = 'unit_number'

for p in [TRAIN_PATH, VALID_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)
test_df = pd.read_csv(TEST_PATH)

for name, df in [('train', train_df), ('valid', valid_df), ('test', test_df)]:
    if TARGET not in df.columns:
        raise ValueError(f"{name}.csv missing target column '{TARGET}'")

feature_cols = [c for c in train_df.columns if c != TARGET]

print(f'Train shape: {train_df.shape}')
print(f'Valid shape: {valid_df.shape}')
print(f'Test  shape: {test_df.shape}')
print(f'Feature count: {len(feature_cols)}')
print(f'ID available for group-aware CV: {ID_COL in train_df.columns}')


### Why this setup
- Uses DS-B as the main tabular feature set.
- Keeps `RUL_clipped` as regression target.
- Checks whether `unit_number` exists for strict group-aware CV.


In [ ]:
def nasa_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    diff = y_pred - y_true
    return float(np.sum(np.where(diff < 0, np.exp(-diff / 13.0) - 1.0,
                                 np.exp(diff / 10.0) - 1.0)))


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


nasa_scorer = make_scorer(nasa_score, greater_is_better=False)
rmse_scorer = make_scorer(rmse, greater_is_better=False)
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)
r2_scorer = make_scorer(r2_score)

X_train = train_df[feature_cols]
y_train = train_df[TARGET]

if ID_COL in train_df.columns:
    groups = train_df[ID_COL]
    cv = GroupKFold(n_splits=5)
    cv_note = 'GroupKFold(5) by unit_number'
else:
    groups = None
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_note = 'KFold(5) fallback (unit_number not present in DS-B export)'

print('CV strategy:', cv_note)


### Metrics and validation policy
- Primary ranking: `RMSE`, then `NASA score`.
- Secondary checks: `MAE`, `R²`.
- CV strategy follows plan constraints where possible; falls back transparently if ID is unavailable.


In [ ]:
models = {
    'KNN': Pipeline([
        ('scaler', MinMaxScaler()),
        ('model', KNeighborsRegressor(n_neighbors=9))
    ]),
    'SVM': Pipeline([
        ('scaler', MinMaxScaler()),
        ('model', SVR(kernel='rbf', C=100, gamma=0.5, epsilon=0.01))
    ]),
    'RandomForest': RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=1,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ),
    'LightGBM': lgb.LGBMRegressor(
        objective='regression',
        random_state=42,
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9
    ),
}

cv_rows = []
for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups,
        cv=cv,
        scoring={'rmse': rmse_scorer, 'mae': mae_scorer, 'r2': r2_scorer, 'nasa': nasa_scorer},
        n_jobs=-1,
        return_train_score=False,
    )

    cv_rows.append({
        'model': name,
        'cv_rmse': -scores['test_rmse'].mean(),
        'cv_mae': -scores['test_mae'].mean(),
        'cv_r2': scores['test_r2'].mean(),
        'cv_nasa': -scores['test_nasa'].mean(),
    })

cv_table = pd.DataFrame(cv_rows).sort_values(['cv_rmse', 'cv_nasa']).reset_index(drop=True)
print('Cross-validation leaderboard:')
display(cv_table)


### Baseline benchmark (Source A + B blend)
- KNN/SVM/RF mirror Source A style and hyperparameters.
- LightGBM adds Source B’s boosting-based tabular benchmark.
- All are scored under one consistent CV protocol.


In [ ]:
# Source-B-style LightGBM tuning (search-based)
lgb_base = lgb.LGBMRegressor(objective='regression', random_state=42)
param_dist = {
    'n_estimators': [200, 300, 500, 700],
    'learning_rate': [0.01, 0.03, 0.05, 0.08],
    'num_leaves': [15, 31, 63, 127],
    'max_depth': [-1, 5, 8, 12],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_samples': [10, 20, 40, 60],
}

search = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)
search.fit(X_train, y_train, groups=groups)

best_lgbm = search.best_estimator_
print('Best LightGBM params:')
print(search.best_params_)
print(f"Best tuned CV RMSE: {-search.best_score_:.4f}")


### LightGBM tuning note
- Uses randomized search to keep compute practical.
- RMSE is the search objective, consistent with plan acceptance metrics.


In [ ]:
X_valid = valid_df[feature_cols]
y_valid = valid_df[TARGET]

holdout_rows = []
pred_store = {}

for name, model in {**models, 'LightGBM_tuned': best_lgbm}.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    holdout_rows.append({
        'model': name,
        'valid_rmse': rmse(y_valid, y_pred),
        'valid_mae': mean_absolute_error(y_valid, y_pred),
        'valid_r2': r2_score(y_valid, y_pred),
        'valid_nasa': nasa_score(y_valid, y_pred),
    })
    pred_store[name] = y_pred

holdout_table = pd.DataFrame(holdout_rows).sort_values(['valid_rmse', 'valid_nasa']).reset_index(drop=True)
champion_name = holdout_table.iloc[0]['model']
print('Holdout leaderboard:')
display(holdout_table)
print(f'Champion model: {champion_name}')


### Holdout interpretation
- Validation ranking uses the same metric family as CV.
- Champion is selected by lowest RMSE with NASA score as safety-oriented tie-break context.


In [ ]:
# Source-A-style visual diagnostics: scatter + identity + sequence plot
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()
plot_models = ['KNN', 'SVM', 'RandomForest', 'LightGBM_tuned']

for ax, m in zip(axes, plot_models):
    yp = pred_store[m]
    ax.scatter(y_valid, yp, s=12, alpha=0.35)
    mn = min(float(y_valid.min()), float(yp.min()))
    mx = max(float(y_valid.max()), float(yp.max()))
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.2)
    ax.set_title(f'{m}: Actual vs Predicted')
    ax.set_xlabel('Actual RUL')
    ax.set_ylabel('Predicted RUL')

plt.tight_layout()
plt.show()

# Sequence-style view for champion (sampled window for readability)
champ_pred = pred_store[champion_name]
n_show = min(250, len(y_valid))

plt.figure(figsize=(12, 4))
plt.plot(np.arange(n_show), y_valid.values[:n_show], label='Actual', color='gray')
plt.plot(np.arange(n_show), champ_pred[:n_show], label=f'Predicted ({champion_name})', color='#2E75B6')
plt.title(f'Champion prediction trace (first {n_show} validation rows)')
plt.xlabel('Row index')
plt.ylabel('RUL')
plt.legend()
plt.tight_layout()
plt.show()

# Residual distribution for champion
residuals = y_valid.values - champ_pred
plt.figure(figsize=(7, 4))
sns.histplot(residuals, bins=40, kde=True, color='#70AD47')
plt.title(f'Residual distribution — {champion_name}')
plt.xlabel('Residual (actual - pred)')
plt.tight_layout()
plt.show()


### Diagnostic rationale
- Scatter + identity line checks calibration and bias.
- Trace plot checks temporal consistency behavior in a Source A style.
- Residual histogram highlights skew and spread.


In [ ]:
# Feature importance for tree-based champions when available
if champion_name in ['RandomForest', 'LightGBM', 'LightGBM_tuned']:
    champion_model = best_lgbm if champion_name == 'LightGBM_tuned' else models[champion_name]
    if champion_name != 'LightGBM_tuned':
        champion_model.fit(X_train, y_train)

    if hasattr(champion_model, 'feature_importances_'):
        imp_df = pd.DataFrame({
            'feature': feature_cols,
            'importance': champion_model.feature_importances_
        }).sort_values('importance', ascending=False)

        plt.figure(figsize=(8, 6))
        sns.barplot(data=imp_df.head(20), x='importance', y='feature', color='#1F4E79')
        plt.title(f'Top 20 feature importances ({champion_name})')
        plt.tight_layout()
        plt.show()

        display(imp_df.head(20))

summary_payload = {
    'cv_strategy': cv_note,
    'champion_model': str(champion_name),
    'best_valid_rmse': float(holdout_table.iloc[0]['valid_rmse']),
    'best_valid_mae': float(holdout_table.iloc[0]['valid_mae']),
    'best_valid_r2': float(holdout_table.iloc[0]['valid_r2']),
    'best_valid_nasa': float(holdout_table.iloc[0]['valid_nasa']),
}
print('Modeling checkpoint:')
print(summary_payload)


## Transition to Ensemble Notebook
This notebook now provides:
- Baseline + tuned model leaderboard.
- Champion model evidence using RMSE, MAE, R², NASA score.
- Source A/B aligned plots and diagnostics for handoff to ensemble/selection.
